# 02 – Preprocess SD7K (Kaggle / local CPU)
Resize SD7K pairs to 512×512 JPEG (quality 95) and split by document ID.

**Run this on Kaggle**, then save output as a Kaggle Dataset `sd7k-512`.

Expected raw folder layout:
```
sd7k_raw/
  train/input/      ← shadow images
  train/target/     ← shadow-free images
  test/input/
  test/target/
```

In [ ]:
import os, sys, shutil
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
import random
from PIL import Image
import numpy as np
from tqdm.auto import tqdm

# ── Paths ──────────────────────────────────────────────────────────────────
KAGGLE = os.path.exists('/kaggle')
if KAGGLE:
    # Attach your raw SD7K as a Kaggle Dataset first, then set:
    SD7K_RAW = Path('/kaggle/input/sd7k-raw')   # <- change to your dataset name
    OUT_DIR  = Path('/kaggle/working/sd7k_512')
else:
    SD7K_RAW = Path('data/sd7k_raw')
    OUT_DIR  = Path('data/sd7k_512')

IMAGE_SIZE = 512
JPEG_Q     = 95
VAL_FRAC   = 0.05
SEED       = 42

print(f'SD7K raw: {SD7K_RAW}')
print(f'Output:   {OUT_DIR}')

In [ ]:
# ── Inspect folder structure ───────────────────────────────────────────────
# Expected:
#   SD7K_RAW/train/input/          <doc_id>_<occluder_id>.jpg
#   SD7K_RAW/train/target/
#   SD7K_RAW/test/input/
#   SD7K_RAW/test/target/

for split in ['train', 'test']:
    for subdir in ['input', 'target']:
        p = SD7K_RAW / split / subdir
        n = len(list(p.glob('*.jpg'))) if p.exists() else 0
        print(f'  {split}/{subdir}: {n} images')

# Show a few filenames to understand naming convention
sample_files = list((SD7K_RAW / 'train' / 'input').glob('*.jpg'))[:5]
print('\nSample filenames:', [f.name for f in sample_files])

In [ ]:
# ── Extract document IDs from filenames ───────────────────────────────────
# IMPORTANT: Split by document ID, not by image, to avoid data leakage.
# Adjust the regex below to match the actual filename format.
import re

def get_doc_id(filename: str) -> str:
    """Extract document ID from filename.
    
    Common SD7K formats:
      '0001_001.jpg'  → doc_id = '0001'
      'doc001_cap01.jpg' → doc_id = 'doc001'
    Adjust the pattern to match your download.
    """
    # Try pattern: <doc_id>_<anything>.ext
    m = re.match(r'^(.+?)_', filename)
    return m.group(1) if m else filename
    
train_input_dir = SD7K_RAW / 'train' / 'input'
all_files = sorted(f.name for f in train_input_dir.glob('*.jpg'))
doc_ids = sorted(set(get_doc_id(f) for f in all_files))
print(f'Found {len(doc_ids)} unique document IDs in training set')
print(f'First 10 doc IDs: {doc_ids[:10]}')

In [ ]:
# ── Train/Val split by document ID ────────────────────────────────────────
random.seed(SEED)
n_val_docs = max(1, int(len(doc_ids) * VAL_FRAC))
val_docs = set(random.sample(doc_ids, n_val_docs))
train_docs = set(doc_ids) - val_docs

print(f'Train docs: {len(train_docs)}, Val docs: {len(val_docs)}')

train_files = [f for f in all_files if get_doc_id(f) in train_docs]
val_files   = [f for f in all_files if get_doc_id(f) in val_docs]
print(f'Train images: {len(train_files)}, Val images: {len(val_files)}')

In [ ]:
# ── Resize and save ────────────────────────────────────────────────────────
def resize_pair(args):
    fn, src_split, dst_split, src_root, dst_root = args
    for subdir in ['input', 'target']:
        src = src_root / src_split / subdir / fn
        dst = dst_root / dst_split / subdir / fn
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists():
            return
        with Image.open(src).convert('RGB') as img:
            img_resized = img.resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)
            img_resized.save(dst, 'JPEG', quality=JPEG_Q)

tasks = []
# Training split
tasks += [(fn, 'train', 'train', SD7K_RAW, OUT_DIR) for fn in train_files]
# Validation split
tasks += [(fn, 'train', 'val', SD7K_RAW, OUT_DIR) for fn in val_files]
# Official test set (keep as-is)
test_files = sorted(f.name for f in (SD7K_RAW / 'test' / 'input').glob('*.jpg'))
tasks += [(fn, 'test', 'test', SD7K_RAW, OUT_DIR) for fn in test_files]

print(f'Total pairs to process: {len(tasks)}')

with ProcessPoolExecutor(max_workers=4) as ex:
    list(tqdm(ex.map(resize_pair, tasks), total=len(tasks), desc='Resizing'))

print('Done! Folder sizes:')
for split in ['train', 'val', 'test']:
    n = len(list((OUT_DIR / split / 'input').glob('*.jpg')))
    print(f'  {split}: {n} pairs')

In [ ]:
# ── Quick sanity check ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt
sample = list((OUT_DIR / 'train' / 'input').glob('*.jpg'))[:3]
fig, axes = plt.subplots(2, 3, figsize=(10, 7))
for col, sp in enumerate(sample):
    s = Image.open(sp)
    c = Image.open(OUT_DIR / 'train' / 'target' / sp.name)
    axes[0, col].imshow(s); axes[0, col].set_title(sp.name[:12], fontsize=7); axes[0, col].axis('off')
    axes[1, col].imshow(c); axes[1, col].axis('off')
plt.suptitle(f'Preprocessed pairs ({IMAGE_SIZE}×{IMAGE_SIZE})')
plt.tight_layout(); plt.show()

In [ ]:
# ── (Optional) Zip output for easy download / Kaggle dataset creation ────────
if KAGGLE:
    print('Archiving /kaggle/working/sd7k_512 into zip...')
    shutil.make_archive('/kaggle/working/sd7k_512', 'zip', '/kaggle/working/sd7k_512')
    print('Saved: /kaggle/working/sd7k_512.zip')